# Red Five — folds and trial history

Synthetic development example, not investment evidence. Declare chronological windows, inspect every row's role, then register and evaluate individual sections. No upstream models, residualizers or weights are fitted. Fold timestamps do not prove upstream out-of-sample predictions or independent observations.

In [ ]:
%matplotlib inline
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from matplotlib.figure import Figure

from red_five.component_export import Component, export_components
from red_five.composition import PlotOptions, Selection
from red_five.contracts import ContractError
from red_five.quantiles import QuantileConfig
from red_five.rendering import verify_bundle
from red_five.temporal import FoldSpec, audit_folds
from red_five.trials import TrialLedger

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "examples/quantile-signals.csv").is_file()
)
signals = (ROOT / "examples/quantile-signals.csv").read_bytes()
config = (ROOT / "examples/quantile-evaluation.json").read_bytes()
plan = (ROOT / "STATISTICAL_ANALYSIS_PLAN.md").read_bytes()
lock = (ROOT / "uv.lock").read_bytes()
folds = (
    FoldSpec(
        "first", "2026-01-01T00:00:00Z", "2026-01-21T00:00:00Z", "2026-01-26T00:00:00Z"
    ),
    FoldSpec(
        "second", "2026-01-01T00:00:00Z", "2026-01-26T00:00:00Z", "2026-01-31T00:00:00Z"
    ),
)
audit = audit_folds(signals, config, folds)

## Inspect the fold summary and individual rows separately

Test windows are half-open and nonoverlapping. Eligible training labels must be nonmissing and available **strictly before** test start minus the declared gap. Equality is excluded. Gap, unavailable-label, missing-label and outside-window roles are disjoint. This conservative training filter also applies to quantile boundary fitting. Test labels are scored only when mature by report `as_of`.

In [ ]:
display(audit.summary.dataframe())

In [ ]:
display(
    audit.membership.dataframe().head(30)
)  # Explicit preview, not the full membership

## Register each attempt before evaluating

The local journal retains hashes of the request, exact section evidence and failed/unavailable outcomes. Running this setup again creates a new attempt prefix; rerunning a registered ID fails instead of overwriting it. Registered-only entries indicate unfinished work. Historical search completeness remains unknown; counts here are not an approved multiplicity correction.

In [ ]:
ledger = TrialLedger(ROOT / "build" / "temporal-trials.sqlite")
attempt = uuid4().hex[:12]
results = {}
for fold in folds:
    results[fold.fold_id] = ledger.run(
        f"{attempt}-{fold.fold_id}",
        "synthetic-quantile-family",
        "quantiles",
        signals,
        config,
        plan,
        lock,
        fold=fold,
        quantiles=QuantileConfig(
            fold.test_start, bins=5, minimum_training=15, minimum_bin=1
        ),
    )
display(ledger.table().dataframe())

## One table, one chart, or your own layout

These are held-out decision-window summaries, not certified OOS model performance. Counts of one in this tiny fixture are for software demonstration only, not statistical adequacy. Per-fold sections use the same immutable tables, selection, styling and exports as other reports.

In [ ]:
first = results["first"].select(Selection(model_ids=("ES-model",), precision=4))
display(first)
display(first.figure(options=PlotOptions(title="First fold: ES bin means")))
display(first.figure("quantile_counts"))

In [ ]:
figure = Figure(figsize=(11, 12), layout="constrained")
axes = figure.subplots(2, 1)
for ax, fold in zip(axes, folds, strict=True):
    results[fold.fold_id].select(Selection(model_ids=("ES-model",))).plot(
        ax=ax, options=PlotOptions(title=f"{fold.fold_id}: ES frozen-bin means")
    )
display(figure)

## Failure history stays visible

This deliberately malformed synthetic CSV illustrates failure retention. Only its digest and exception class are recorded, not the error text. Interruptions may leave a registered-only attempt; no automatic retry or hidden deletion occurs.

In [ ]:
try:
    ledger.run(
        f"{attempt}-failure-demo",
        "synthetic-quantile-family",
        "coverage",
        b"deliberately-invalid-synthetic-csv",
        config,
        plan,
        lock,
        fold=folds[0],
    )
except ContractError:
    print("Expected input failure retained in the journal")
display(ledger.table().dataframe())

## Export chosen fold components

These are verified partial section bundles, not a full trial-journal export. The journal remains local and private. Keep it for research history; unlike disposable charts, deleting it loses trial evidence. Local hashes cannot establish complete history or detect a rewritten/truncated journal without an external anchor.

In [ ]:
output = ROOT / "build" / f"fold-components-{attempt}"
export_components(
    [
        Component(
            results[f.fold_id].select(Selection(model_ids=("ES-model",))),
            "quantiles",
            PlotOptions(title=f"{f.fold_id}: ES means"),
        )
        for f in folds
    ],
    output,
)
assert verify_bundle(output)["scope"] == "partial"
print(output)

Next gates: upstream fold-local training provenance, nested selection/final assessment policy, dependence-aware uncertainty and a declared hypothesis family/correction. No p-values or verdict are produced. See `docs/TEMPORAL_TRIALS.md`.